## Задание 1
Потренируйтесь вычислять размер модели в памяти. Это поможет понять требования к оборудованию. 

Используйте алгоритм, описанный выше. Посчитайте: 
- Количество параметров и размер каждого элемента. Для подсчёта количества параметров используйте `sum(p.numel() for p in model.parameters())`.
- Размер буферов модели.
- Суммарный размер переведите в МБ.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModel
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()


def get_model_size_mb(model):
    # Считаем размер всех параметров модели
    param_size = 0
    for param in model.parameters():
        # Количество элементов * размер каждого элемента
        param_size += param.numel() * param.element_size()
    
    # Считаем размер буферов модели (например, batch norm statistics)
    buffer_size = 0
    for buffer in model.buffers():
        # Аналогично параметрам
        buffer_size += buffer.numel() * buffer.element_size()
    
    # Переводим из байт в мегабайты
    # (param_size + buffer_size) делить на 1024 дважды
    total_size_mb = (param_size + buffer_size) / 1024 / 1024
    return total_size_mb

# Тестируем функцию на разных моделях
models_to_test = [
    "distilbert-base-uncased",
    "cointegrated/rubert-tiny",
    "microsoft/MiniLM-L12-H384-uncased"
]

print("Сравнение размеров моделей")
print("=" * 40)

for model_name in models_to_test:
    print(f"\nЗагружаем {model_name}...")
    
    # Загружаем модель
    model = AutoModel.from_pretrained(model_name, ignore_mismatched_sizes=True)
    
    # Вычисляем размер
    size_mb = get_model_size_mb(model)
    
    # Считаем количество параметров
    # Сумма всех параметров модели в миллионах
    num_params = sum(p.numel() for p in model.parameters())
    
    print(f"Размер: {size_mb:.1f} МБ")
    print(f"Параметры: {num_params/10**6:1f}М")
    print("-" * 40)

/Users/papa/Documents/Практикум/Deep learning Engineer/dle_practicum/Sprint 5/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Сравнение размеров моделей

Загружаем distilbert-base-uncased...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5113.76it/s]


Размер: 253.2 МБ
Параметры: 66.362880М
----------------------------------------

Загружаем cointegrated/rubert-tiny...


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 7742.98it/s]


Размер: 45.0 МБ
Параметры: 11.784168М
----------------------------------------

Загружаем microsoft/MiniLM-L12-H384-uncased...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 33187.53it/s]

Размер: 127.3 МБ
Параметры: 33.360000М
----------------------------------------


## Задание 2
Потренируйтесь корректно измерять скорость работы моделей. 

Реализуйте самостоятельно функцию `measure_inference_time`. Она принимает на вход три параметра: `classifier` — объект пайплайна для классификации текста, `texts` — список строк с тестовыми текстами, `num_runs` (по умолчанию 10) — число полных прогонов по всем текстам для усреднения.

Для каждого прогона функция должна: 
- запоминать текущее время перед циклом по всем текстам,
- вызывать `classifier(text)` для каждого текста из texts,
- по завершении всех текстов измерять время ещё раз и сохранять затраченное время в список.

После всех прогонов нужно вычислить среднее время одного прогона, разделить его на число текстов, перевести в миллисекунды и вернуть результат.

In [5]:
import time
from transformers import pipeline
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

def measure_inference_time(classifier, texts, num_runs=10):
    times = []
    for _ in range(num_runs):
        start_time = time.time()
        # Прогоняем все тексты через классификатор
        for text in texts:
            _ = classifier(text)
        end_time = time.time()
        # Сохраняем время выполнения
        times.append(end_time - start_time)
    
    avg_time = sum(times) / len(times)
    # Среднее время на один текст
    avg_time_per_text = avg_time / len(texts)
    return avg_time_per_text * 1000  # в миллисекундах

# Тестовые тексты на английском
test_texts = [
    "This product is amazing!", # Этот продукт потрясающий!
    "Terrible quality, very disappointed.", # Ужасное качество, очень разочарован.
    "The service was okay, nothing special.", # Сервис был нормальным, ничего особенного.
    "Outstanding experience! Highly recommend!", # Выдающийся опыт! Настоятельно рекомендую!
    "Poor customer support, took forever." # Плохая поддержка клиентов, всё заняло вечность.
]

# Список англоязычных моделей для тестирования
models_to_test = [
    "distilbert-base-uncased",
    "microsoft/MiniLM-L12-H384-uncased",
    "cointegrated/rubert-tiny",
]

for model_name in models_to_test:
    print(f"\nТестирование скорости: {model_name}")
    print("=" * 50)
    
    # Создаём pipeline для модели
    classifier = pipeline("text-classification", model=model_name)
    
    # Измеряем среднее время инференса
    avg_time = measure_inference_time(classifier, test_texts, 100)
    
    print(f"Среднее время обработки одного текста: {avg_time:.3f} мс")


Тестирование скорости: distilbert-base-uncased


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6871.85it/s]


Среднее время обработки одного текста: 12.651 мс

Тестирование скорости: microsoft/MiniLM-L12-H384-uncased


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 39180.70it/s]


Среднее время обработки одного текста: 12.833 мс

Тестирование скорости: cointegrated/rubert-tiny


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 5916.41it/s]


Среднее время обработки одного текста: 6.315 мс
